# 🛡️ FraudShield AI: End-to-End Cyber Defense Platform
**Team**: Hackscribe  
**Hackathon Theme**: Future of Fraud Detection and Prevention  

FraudShield AI operates across two core operational pipelines:
1. **Prevent Gate**: Real-time multi-modal pre-publish gate evaluating optical text, rogue URLs, embedded QR codes, and psychological manipulation before fraudulent communications reach citizens.
2. **Hunt Mode**: Autonomous background threat sweeper monitoring public feeds (OpenPhish, URLhaus) and lookalike domains to intercept malicious infrastructure proactively.

---
## Step 2: Load and Clean SMS Spam Collection Dataset
- **Source**: UCI Machine Learning Repository
- **Goal**: Clean text data, encode binary labels (`0 = Safe`, `1 = Fraud`), remove duplicate messages, and structure data for NLP.

In [ ]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[["v1", "v2"]]
df.columns = ["label", "text"]
df["label"] = df["label"].map({"ham": 0, "spam": 1})
df = df.drop_duplicates().reset_index(drop=True)
df.to_csv("data/clean_sms.csv", index=False)

print(f"Total clean samples: {len(df)}")
print(df["label"].value_counts())
df.head()

---
## Step 3: TF-IDF Vectorizer + Logistic Regression Classifier
- **Why TF-IDF?** Converts raw English text into numeric word-importance scores.
- **Why n-grams (1, 2)?** Detects multi-word scam triggers like *"claim now"*, *"account suspended"*.
- **Why Balanced Class Weights?** In fraud detection, missing a scam (False Negative) is dangerous. Class weighting penalizes missed fraud heavily.

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df = pd.read_csv("data/clean_sms.csv")
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.20, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=4000, sublinear_tf=True, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print("Text Model Evaluation:")
print(classification_report(y_test, y_pred, target_names=["Safe", "Fraud"]))

---
## Step 4: 16 Lexical URL Features + Random Forest Classifier
- **16 Engineered Lexical Features**: Length metrics, entropy, suspicious TLD detection, IP addressing, keyword tokens, brand impersonation markers.
- **Random Forest**: Ensemble of 100 decision trees to detect evasive phishing and scam landing pages.

In [ ]:
from step4_train_url_model import extract_url_features
import joblib

sample_urls = [
    "https://google.com/search?q=cybersecurity",
    "http://192.168.1.100/sbi/verify-kyc.php",
    "http://secure-update-hdfcbank.account-service.top/login"
]

clf = joblib.load("models/url_classifier.pkl")

for u in sample_urls:
    feats = extract_url_features(u)
    prob = clf.predict_proba([feats])[0][1]
    print(f"URL: {u}")
    print(f"Phishing Probability: {prob:.4f} ({'SUSPICIOUS' if prob > 0.4 else 'BENIGN'})
")

---
## Step 5: Multi-Modal Tri-Signal Risk Engine
- Combines **Text NLP Model (35%)**, **Lexical URL Classifier (35%)**, and **Heuristic / Manipulation Sandbox (30%)**.
- Outputs **Allow / Hold / Block** verdicts with calibrated risk index (0 - 100).

In [ ]:
from risk_engine import FraudShieldRiskEngine
import json

engine = FraudShieldRiskEngine()
result = engine.analyze(
    text="Dear customer, your electricity power will be disconnected tonight at 9:30 PM. Contact officer immediately.",
    url="http://sbi-kyc-update.xyz/login.php"
)
print(f"Risk Score: {result['risk_score']} / 100")
print(f"Risk Tier:  {result['risk_tier']}")
print(f"Verdict:    {result['verdict']}")
print(f"Scam Type:  {result['scam_type']}")
print(f"Advice:     {result['safe_action_advice']}")

---
## Step 6: Explainable AI (SHAP TreeExplainer & Token Attributions)
- **SHAP (SHapley Additive exPlanations)** on URL Random Forest: Quantifies exact marginal contribution of each structural feature.
- **Linear Attributions on NLP Classifier**: Pinpoints high-risk urgency and coercion keywords triggering fraud classification.
- **Auditor-Ready Transparency**: Explains *why* content was flagged to eliminate black-box AI decisions.

In [ ]:
from explainability import FraudShieldExplainer

explainer = FraudShieldExplainer()
xai_report = explainer.explain_prediction(
    text="Urgent: Your SBI account is frozen! Click to verify PAN card within 24 hours.",
    url="http://sbi-kyc-verification.top/update.html",
    url_risk_prob=0.88,
    text_risk_prob=0.92
)

print("Executive Summary:")
for bullet in xai_report["executive_summary"]:
    print(f"• {bullet}")

print("
Top URL Risk Drivers (SHAP TreeExplainer):")
for f in xai_report["url_explainability"]["features"][:3]:
    print(f"  {f['label']}: {f['direction']} (SHAP: {f['shap_value']:+.4f})")

print("
Top Text Token Drivers:")
for t in xai_report["text_explainability"]["tokens"][:3]:
    print(f"  '{t['token']}': {t['direction']} (+{t['percentage']}%)")

---
## Step 7: Universal AI Image & Screenshot Threat Analyzer
- **Analyzes ANY Image**: WhatsApp chats, fake payment receipts, UPI QR codes, bank KYC notices, police warrants, and clean non-text photos.
- **Dual Optical Pipeline**:
  - **Native Windows OCR (`winocr` / Windows.Media.Ocr)**: Instant optical text extraction.
  - **OpenCV (`cv2.QRCodeDetector`)**: Decodes embedded UPI payment payloads (`upi://pay`) and rogue URLs.
- **Entity Extraction**: Auto-extracts contact phones, rogue URLs, UPI IDs, amounts, and impersonated brands.
- **Clean Image Safety**: Returns 0.0% risk and `ALLOW` verdict for clean, benign documents or photos.

In [ ]:
from image_analyzer import UniversalImageAnalyzer

analyzer = UniversalImageAnalyzer()

samples = [
    ("Electricity Cut Flyer", "sample_scam_poster.png"),
    ("SBI Phishing Screenshot", "sample_sbi_screenshot.png"),
    ("UPI QR Lottery Scam", "sample_upi_qr_scam.png"),
    ("Clean Campus Notice", "sample_clean_photo.png")
]

for label, filename in samples:
    with open(filename, "rb") as f:
        res = analyzer.analyze_image(engine, f.read())
    meta = res["image_metadata"]
    ra = res["risk_analysis"]
    qr = res["qr_code"]
    ent = res["detected_entities"]
    print(f"\n[{label}] -> {filename}")
    print(f"  Category:   {meta['image_category']}")
    print(f"  Verdict:    {ra['verdict']} (Risk: {ra['risk_score']}%)")
    print(f"  QR Payload: {qr['payload'] if qr['has_qr'] else 'None'}")
    print(f"  Phones:     {ent['phones']}")
    print(f"  Brands:     {ent['impersonated_brands']}")
    print(f"  UPI IDs:    {ent['upi_ids']}")
    print(f"  Amounts:    {ent['amounts']}")

---
## Step 8 & 9: Autonomous Threat Hunt Mode & Enterprise API
- **Hunt Mode**: Sweeps live OpenPhish and lookalike domain registries to blocklist scams before distribution.
- **SQLite Audit Trail**: Stores all scan incidents with timestamps and risk scores.

In [ ]:
from hunt_engine import run_hunt_scan
from database import get_recent_incidents, get_stats

# Run Hunt Scan
threats = run_hunt_scan(engine, max_items=5)
print(f"Hunted {len(threats)} live threat targets:")
for t in threats:
    print(f"[{t['verdict']}] ({t['risk_score']}%) -> {t['url']}")

# Audit Stats
print("\nDatabase Audit Stats:", get_stats())